In [1]:
import numpy as np

### 합성곱/풀링 계층 구현하기

In [10]:
x = np.random.rand(10,1,28,28)

In [12]:
x.shape

(10, 1, 28, 28)

In [13]:
x[0].shape

(1, 28, 28)

In [21]:
x[9][0]   # 10번째 데이터, 첫 번째 채널

array([[0.16604355, 0.33280828, 0.46910379, 0.41476693, 0.11019321,
        0.79665342, 0.20316159, 0.46225755, 0.20926393, 0.53629125,
        0.29446902, 0.11046715, 0.13467533, 0.31184509, 0.89024642,
        0.12159699, 0.43662161, 0.70876283, 0.85377149, 0.70801537,
        0.57922911, 0.53366294, 0.63285829, 0.30445139, 0.17863811,
        0.0351668 , 0.4283928 , 0.70100206],
       [0.35175607, 0.86849934, 0.24410865, 0.64518451, 0.75011372,
        0.23671854, 0.36668338, 0.6001731 , 0.58012312, 0.67393689,
        0.56609734, 0.58955231, 0.78099397, 0.39980894, 0.12115656,
        0.28791026, 0.78545512, 0.92909335, 0.77334284, 0.154572  ,
        0.98462137, 0.32675596, 0.09739502, 0.61949705, 0.22624085,
        0.47320172, 0.93980727, 0.06177973],
       [0.63510805, 0.65570501, 0.78869799, 0.10925207, 0.86169503,
        0.27895864, 0.40465328, 0.39481694, 0.69754358, 0.4178667 ,
        0.32418142, 0.35793548, 0.92435673, 0.27030764, 0.79429543,
        0.69906562, 0.6732

### im2col로 데이터 전개하기

In [37]:
# im2col은 입력 데이터(특징 맵)를 필터링(가중치 계산)하기 좋게 전개하는(펼치는) 함수이다. 
# 3차원 입력 데이터에 im2col을 적용하면 2차원 행렬로 바뀐다. 배치 안의 데이터 수까지 포함하여 4차원 데이터를 2차원으로 변환한다.

def im2col(input_data, filter_h, filter_w, stride=1, pad=0): # input_data: (데이터 수, 채널 수, 높이, 너비)의 4차원 배열로 이루어진 입력 데이터, filter_h: 필터의 높이, filter_w: 필터의 너비, stride: 스트라이드, pad: 패딩
    N, C, H, W = input_data.shape               # N: 데이터 수, C: 채널 수, H: 높이, W: 너비
    out_h = (H + 2*pad - filter_h)//stride + 1  # 출력 데이터의 높이 OH, fiter_h는 필터의 높이
    out_w = (W + 2*pad - filter_w)//stride + 1  # 출력 데이터의 너비 OW, filter_w는 필터의 너비

    img = np.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant') # 입력 데이터에 pad만큼 0으로 채움
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w)) # 입력 데이터를 2차원 행렬로 전개할 배열 0으로 초기화

    # 입력 이미지 데이터를 슬라이딩 윈도우 방식으로 필터링하여 2차원 배열로 변환
    # 필터의 높이, 너비만큼 이동하면서 필터를 적용
    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col # 2차원 행렬로 전개


In [39]:
x1 = np.random.rand(1, 3, 7, 7) # (데이터 수, 채널 수, 높이, 너비), input_data
col1 = im2col(x1, 5, 5, stride=1, pad=0) # 필터 크기 5x5, 스트라이드 1, 패딩 0
print(col1)

[[0.74106842 0.97656663 0.29375993 0.70578483 0.63216958 0.19707386
  0.01774778 0.11849819 0.19723493 0.88589142 0.94022101 0.35282044
  0.79920684 0.73294678 0.22413541 0.20008003 0.18840841 0.46055867
  0.27051185 0.34801567 0.04694867 0.36892092 0.38410438 0.92175078
  0.58733491 0.13206816 0.24891976 0.76518073 0.81649653 0.25226892
  0.19358995 0.70192682 0.72301127 0.45548008 0.6306368  0.31569998
  0.00526173 0.72137833 0.0191058  0.35919838 0.26756325 0.33096128
  0.4406989  0.1585167  0.07519856 0.35000824 0.85132742 0.61621305
  0.86770467 0.42948344 0.59199024 0.80821595 0.13938672 0.13943071
  0.53760316 0.26738552 0.98280673 0.53421556 0.89960028 0.60054428
  0.79859721 0.82856016 0.86203494 0.57304717 0.59102694 0.22399215
  0.03923059 0.65370463 0.78671469 0.74047009 0.73120583 0.13802534
  0.36916969 0.98340846 0.23173723]
 [0.97656663 0.29375993 0.70578483 0.63216958 0.58903671 0.01774778
  0.11849819 0.19723493 0.88589142 0.51244274 0.35282044 0.79920684
  0.73294678

### 합성곱 계층 구현  

In [41]:
class Convolution:
  def __init__(self,W,b,stride=1,pad=0):
    self.W = W            # 필터(가중치)
    self.b = b            # 편향
    self.stride = stride  # 스트라이드
    self.pad = pad        # 패딩
  
  def forward(self,x):
    FN, C, FH, FW = self.W.shape  # 필터, FN: 필터 개수, C: 채널 수, FH: 필터 높이, FW: 필터 너비
    N, C, H, W = x.shape          # 입력 데이터, N: 데이터 수, C: 채널 수, H: 높이, W: 너비
    out_h = 1 + int((H + 2*self.pad - FH) / self.stride) # 출력 데이터 높이, OH
    out_w = 1 + int((W + 2*self.pad - FW) / self.stride) # 출력 데이터 너비, OW

    col = im2col(x, FH, FW, self.stride, self.pad) # 입력 데이터를 2차원 배열로 전개한 col
    col_W = self.W.reshape(FN, -1).T               # 필터를 reshape를 사용하여 2차원 배열로 전개한 col_W

    out = np.dot(col, col_W) + self.b              # 합성곱 연산 수행 (행렬의 내적)
    out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)  # 출력 데이터를 reshape하여 transpose

    return out

In [44]:
x1 = np.random.rand(3, 3, 7, 7) # (데이터 수, 채널 수, 높이, 너비), input_data
col1 = im2col(x1, 5, 5, stride=1, pad=0) # 필터 크기 5x5, 스트라이드 1, 패딩 0

# Convolution 클래스 사용
W = np.random.rand(3, 3, 5, 5) # (필터 수, 채널 수, 높이, 너비), filter 5x5x3
b = np.random.rand(1)          # 편향
conv1 = Convolution(W, b, stride=1, pad=0) # Convolution 클래스 생성

out1 = conv1.forward(x1) # 순전파
print(out1)

[[[[16.7882867  16.97294163 16.35500756]
   [18.51749641 17.08498375 16.56354325]
   [19.5090545  18.62575901 17.24664011]]

  [[17.19555781 17.08209342 17.82850469]
   [18.87468779 17.74018176 17.38752363]
   [19.08207082 18.46661982 16.83094699]]

  [[18.26100964 19.05357952 17.1129148 ]
   [21.18483759 18.96220613 19.0476222 ]
   [20.11193561 18.97047587 18.6737503 ]]]


 [[[17.4437636  18.96662542 18.37551626]
   [19.09119495 18.76999742 18.86149185]
   [19.36003142 19.49462625 19.62150281]]

  [[19.12790707 17.88554233 18.05444783]
   [18.91420595 17.93681809 17.48181601]
   [17.59650101 18.49396891 18.23987263]]

  [[19.42601294 20.74521334 20.58254486]
   [20.55572416 19.52954613 18.75428801]
   [18.94683356 19.70484461 20.976135  ]]]


 [[[20.09312927 19.84567341 20.47185001]
   [19.23442441 18.14971649 19.02157916]
   [19.84766519 17.9976305  17.45040528]]

  [[19.71377177 19.54573527 20.3495287 ]
   [18.46530032 18.21389528 17.92196647]
   [18.914354   18.68081109 17.44553505